In [1]:
%load_ext ElasticNotebook

Enabled rmm statistics


In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_11_try_0.pickle

In [3]:
%%RecordEvent
%%time
### cell 12 ###

# Compute gap_length vectorized
prev_id = train['id'].shift()
prev_end = train['discourse_end'].shift()
curr_start = train['discourse_start']

# Condition 1: same essay & gap > 1
cond1 = train['id'].eq(prev_id) & (curr_start.sub(prev_end) > 1)
# Condition 2: new essay & not starting at 0
cond2 = train['id'].ne(prev_id) & curr_start.ne(0)

# Calculate both gap values
gap1 = curr_start.sub(prev_end).sub(2)
gap2 = curr_start.sub(1)

# Assign gap_length
train['gap_length'] = np.where(cond1, gap1, np.where(cond2, gap2, np.nan))
# Manually set the very first row
train.loc[0, 'gap_length'] = 7

# Compute gap_end_length without merge
# Identify last discourse_end in each essay
last_end = train.groupby('id')['discourse_end'].transform('last')
# Mask for last rows of each essay
mask_last = train['discourse_end'].eq(last_end)
# Assign gap_end_length only for last rows where there is text after
train['gap_end_length'] = np.where(
    mask_last & train['discourse_end'].lt(train['essay_len']),
    train['essay_len'].sub(train['discourse_end']),
    np.nan
)

CPU times: user 4.13 ms, sys: 20 µs, total: 4.15 ms
Wall time: 4.16 ms


In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_12_try_0.pickle